In [ ]:
# THINK OF SOMETHING TO BE CREATIVE
# MAKE THE CODE SCALABLE 
# OR SUPER ACCURATE

In [6]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from typing import TypedDict, List, Dict
from langchain_groq import ChatGroq
import json

In [ ]:
class AgentState(TypedDict):
    user_profile_path: str
    product_catalog_path: str
    unique_categories: List[str]
    product_catalogue: dict
    user_profile: dict
    justification: str
    recommended_category: str
    top3_products: List[str]
    error: str 

In [17]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or mixtral-8x7b
    temperature=0.6,
    api_key = "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"
)

# give a prompt to take the user history and the unique categories and choose a likeable category
category_prompt = PromptTemplate.from_template("""
You are an intelligent shopping recommendation assistant.

You will receive:
1. The user's purchase history as a JSON list.
2. A list of unique product categories available in the catalog.

Your task:
- Analyze the user's purchase history.
- Identify categories the user has **not purchased from** before.
- If all categories have been purchased from, choose one the user has **purchased from the least**.
- Choose **one category** that the user might be most interested in exploring next.
- Generate a **catchy, persuasive justification** in natural language explaining *why* this category fits the user’s preferences or lifestyle.

Return your final answer **strictly** in the following JSON format:

{{
  "recommended_category": "<category_name>",
  "justification": "<short persuasive message>"
}}

### USER HISTORY:
{user_history}

### AVAILABLE CATEGORIES:
{available_categories}

Now respond only with the JSON object described above — no extra text.
""")
category_chain = LLMChain(llm=llm, prompt=category_prompt)



In [ ]:
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

In [ ]:
#node 1
# add conditional edges
def category_extraction(state ,user_profile_path ,product_catalog_path ):
    state["user_profile_path"] = user_profile_path
    state["product_catalog_path"] = product_catalog_path
    # read user profile
    try:
        with open(state["user_profile_path"], "r") as f:
            state["user_profile"] = json.load(f)
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['user_profile_path']} not found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['user_profile_path']}."
        return state

    # read product catalogue
    try:
        with open(state["product_catalog_path"], "r") as f:
            state["product_catalogue"] = json.load(f)
            state["unique_categories"] = sorted({item["category"] for item in state["product_catalogue"]})
    except FileNotFoundError:
        state["error"] = f"❌ File path: {state['product_catalog_path']} found. Please check the path."
        return state
    except json.JSONDecodeError:
        state["error"] = f"❌ Invalid JSON format in {state['product_catalog_path']}."
        return state
    return state


In [ ]:
# if no error go to node 2
# node 2 
from pydantic import BaseModel, ValidationError

class CategoryRecommendation(BaseModel):
    recommended_category: str = None
    justification: str = None

def category_recommendation(state):
    try:
        # Run the LLM chain with provided user data
        output = chain.run(
            user_history=state["user_profile"]["history"],
            available_categories=state["unique_categories"]
        )

        # Parse JSON
        data = json.loads(output.strip())

        # Validate using your Pydantic model
        validated = CategoryRecommendation(**data)

        # Additional safeguard: ensure both fields are not empty or whitespace
        if not validated.recommended_category.strip() or not validated.justification.strip():
            state["error"] = "❌ One or more fields are empty or invalid in LLM output."
            return state

        # Assign validated values to state
        state["recommended_category"] = validated.recommended_category.strip()
        state["justification"] = validated.justification.strip()


        # CHECK FOR CORRECTNESS

    except json.JSONDecodeError:
        state["error"] = "❌ Invalid JSON format in LLM output."
        return state
    except ValidationError as e:
        state["error"] = f"❌ Invalid LLM output: {str(e)}"
        return state
    except Exception as e:
        state["error"] = f"❌ Unexpected error: {str(e)}"
        return state

    # If everything passes, clear error and return
    return state


In [9]:
graph.add_conditional_edges(
    "recommend_category_node",
    lambda state: "error_handler" if "error" in state else "recommend_products_node"
)

✅ Loaded user profile for user_id: A123
{'user_id': 'A123', 'history': [{'product': 'Wireless Headphones X200', 'category': 'electronics', 'price': 199.99}, {'product': 'Smartwatch Fit S3', 'category': 'electronics', 'price': 249.99}, {'product': 'Running Shoes SprintX', 'category': 'sportswear', 'price': 89.99}, {'product': 'Sports Backpack 25L', 'category': 'sportswear', 'price': 49.99}, {'product': 'Fitness Tracker Watch', 'category': 'sportswear', 'price': 99.99}, {'product': 'Mystery Novel: Hidden Truths', 'category': 'books', 'price': 14.99}, {'product': 'Science Fiction: Galactic Wars', 'category': 'books', 'price': 19.99}, {'product': 'Cookbook: World Cuisines', 'category': 'books', 'price': 29.99}, {'product': 'Thriller: The Last Escape', 'category': 'books', 'price': 22.99}, {'product': 'Data Science Handbook', 'category': 'books', 'price': 59.99}]}


In [7]:
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplatep

# Create the LLM instance
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # or mixtral-8x7b
    temperature=0.6,
    api_key = "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"
)

prompt = PromptTemplate.from_template("Recommend a product for someone who likes {category}")
chain = LLMChain(llm=llm, prompt=prompt)

print(chain.run(category="electronics"))

For someone who likes electronics, I'd recommend the following product:

**Raspberry Pi 4 Model B**

The Raspberry Pi 4 Model B is a compact, affordable, and highly versatile single-board computer that's perfect for electronics enthusiasts. It's a great tool for learning programming, experimenting with DIY projects, and building innovative devices.

Here are some features that make it an excellent choice:

1. **Quad-core processor**: The Raspberry Pi 4 Model B comes with a powerful quad-core Cortex-A72 CPU, which provides a significant boost in performance compared to its predecessors.
2. **4GB or 8GB RAM**: You can choose from two RAM options, depending on your needs. The 4GB model is perfect for general use, while the 8GB model is ideal for more demanding applications.
3. **Dual-display support**: The Raspberry Pi 4 Model B can support two 4K displays at 30Hz, making it perfect for multimedia projects, gaming, and more.
4. **Gigabit Ethernet**: The board features a built-in gigabit E

In [ ]:
groq_api = "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"

In [ ]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["text"],
    template="""
Extract the following fields from the text below and return them as a VALID JSON object only, with no extra commentary:

Schema:
{{
  "company_name": "string",
  "founded_year": "integer",
  "ceo": "string",
  "headquarters": "string"
}}

Text:
{text}

Rules:
- Return only valid JSON.
- Use null if information is missing.
- founded_year must be an integer if present.
- Do not include additional fields.
"""
)


In [5]:
from groq import Groq

client = Groq(api_key="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")

models = client.models.list()

for model in models.data:
    print(model.id)

playai-tts-arabic
playai-tts
qwen/qwen3-32b
meta-llama/llama-4-maverick-17b-128e-instruct
allam-2-7b
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
openai/gpt-oss-20b
whisper-large-v3-turbo
moonshotai/kimi-k2-instruct
whisper-large-v3
groq/compound-mini
llama-3.3-70b-versatile
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-prompt-guard-2-22m
llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
groq/compound
meta-llama/llama-guard-4-12b


In [ ]:
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class State(BaseModel):
    company_name: str | None = Field(None)
    founded_year: int | None = Field(None)
    ceo: str | None = Field(None)
    headquarters: str | None = Field(None)

parser = PydanticOutputParser(pydantic_object=CompanyInfo)

In [ ]:
from langchain.chains import LLMChain

llm = ChatOpenAI(temperature=0)

chain = LLMChain(
    llm=llm,
    prompt=prompt_template,
    output_parser=parser
)
text = "Acme Corp was founded in 1998 and is currently led by CEO John Doe. Its headquarters are in New York City."

result = chain.run(text=text)
print(result)


In [ ]:
agent = initialize_agent(
    tools,
    llm,
    agent_type="zero-shot-react-description",
    verbose=True
)

response = agent.run("")

In [ ]:
# Create the graph
graph = StateGraph(ShoppingState)

# Add nodes
graph.add_node("category_agent", category_agent)
graph.add_node("product_agent", product_agent)

# Define edges (category -> product -> END)
graph.add_edge("category_agent", "product_agent")
graph.add_edge("product_agent", END)

# Set the entry point
graph.set_entry_point("category_agent")

# Compile the graph
shopping_graph = graph.compile()

In [ ]:
"""
Modular Shopping Assistant Agent
================================

This single-file implementation provides a modular AI agent that:
1. Accepts buyer history (JSON-like dict)
2. Uses an LLM interface (or a simple heuristic fallback) to recommend a product category
3. Searches a mock product catalog and returns the top-3 products
4. Simulates purchase and records a transaction in a mock history log
5. Persists updated buyer history to local JSON memory

How it is organized
-------------------
- Catalog: a small in-memory product catalog with a loading helper
- MemoryStore: read/write buyer profiles and transaction logs to JSON files
- Recommender: LLMRecommender class with two modes:
    * `openai` mode (placeholder stub) — if you wire an LLM, plug it in
    * `heuristic` mode — rule-based recommendation when no LLM is available
- ShoppingAgent: orchestrates the pipeline: analyze -> recommend -> search -> rank -> purchase

Run the demo at the bottom with `python modular_shopping_agent.py`.

Notes
-----
- This file is intentionally self-contained and framework-agnostic.
- If you want real LLM recommendations, replace `LLMRecommender._call_llm` with a call
  to your LLM provider (OpenAI, local LLM, etc.).

"""
from __future__ import annotations

import json
import math
import random
import datetime
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

In [ ]:
# -----------------------------
# Data classes
# -----------------------------

@dataclass
class Product:
    id: str
    name: str
    category: str
    price: float
    rating: float = 4.0
    popularity: int = 100  # arbitrary popularity score


@dataclass
class Transaction:
    user_id: str
    product_id: str
    product_name: str
    price: float
    timestamp: str


# -----------------------------
# Mock Catalog
# -----------------------------

class Catalog:
    def __init__(self, products: Optional[List[Product]] = None):
        self.products: List[Product] = products or self._default_products()

    @staticmethod
    def _default_products() -> List[Product]:
        # example catalog (expand as needed)
        return [
            Product("p1", "Bluetooth Headphones X100", "electronics", 120.0, 4.3, 320),
            Product("p2", "Wireless Earbuds Lite", "electronics", 75.0, 4.1, 540),
            Product("p3", "Running Shoes Pro", "sportswear", 85.0, 4.5, 410),
            Product("p4", "Trail Running Shoes", "sportswear", 110.0, 4.2, 210),
            Product("p5", "Smartwatch S", "electronics", 199.0, 4.6, 190),
            Product("p6", "Yoga Mat Classic", "sportswear", 25.0, 4.0, 150),
            Product("p7", "Noise-Cancelling Headphones Z", "electronics", 250.0, 4.7, 430),
            Product("p8", "Cycling Jersey", "sportswear", 40.0, 3.9, 120),
            Product("p9", "Portable Charger 20000mAh", "electronics", 35.0, 4.1, 480),
            Product("p10", "Fitness Tracker Mini", "electronics", 55.0, 4.0, 300),
        ]

    def find_by_category(self, category: str) -> List[Product]:
        return [p for p in self.products if p.category.lower() == category.lower()]


# -----------------------------
# Memory / Persistence
# -----------------------------

class MemoryStore:
    def __init__(self, filepath: str = "memory.json", tx_filepath: str = "transactions.json"):
        self.filepath = Path(filepath)
        self.tx_filepath = Path(tx_filepath)
        if not self.filepath.exists():
            self._write_json({})
        if not self.tx_filepath.exists():
            self._write_json([])

    def _read_json(self, path: Path) -> Any:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)

    def _write_json(self, data: Any):
        with self.filepath.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def read_profile(self, user_id: str) -> Optional[Dict[str, Any]]:
        data = self._read_json(self.filepath)
        return data.get(user_id)

    def write_profile(self, user_id: str, profile: Dict[str, Any]):
        data = self._read_json(self.filepath)
        data[user_id] = profile
        self._write_json(data)

    def append_transaction(self, tx: Transaction):
        data = self._read_json(self.tx_filepath)
        data.append(asdict(tx))
        with self.tx_filepath.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def list_transactions(self, user_id: Optional[str] = None) -> List[Dict[str, Any]]:
        data = self._read_json(self.tx_filepath)
        if user_id is None:
            return data
        return [t for t in data if t.get("user_id") == user_id]


# -----------------------------
# Recommender (LLM wrapper + heuristic)
# -----------------------------

class LLMRecommender:
    """A small wrapper that either calls an external LLM (placeholder) or uses a heuristic.

    To integrate a real LLM, replace `_call_llm` with a proper API call and keep the
    prompt formatting in `recommend_category`.
    """

    def __init__(self, mode: str = "heuristic"):
        assert mode in ("heuristic", "openai"), "mode must be 'heuristic' or 'openai'"
        self.mode = mode

    def recommend_category(self, user_profile: Dict[str, Any]) -> Tuple[str, str]:
        """Return (recommended_category, justification_text)"""
        if self.mode == "openai":
            # Placeholder: you'd implement a real LLM call here.
            resp = self._call_llm(user_profile)
            # Expecting a small structured response; here we just assume free text and parse minimally.
            # For demonstration, we just echo a plausible choice.
            return resp.get("category", "electronics"), resp.get("justification", "")
        else:
            return self.heuristic_recommend(user_profile)

    def heuristic_recommend(self, user_profile: Dict[str, Any]) -> Tuple[str, str]:
        history = user_profile.get("history", [])
        if not history:
            # New user fallback
            return (
                "electronics",
                "No purchase history found; recommending electronics because it has broad appeal and offers many cross-sell opportunities.",
            )

        # Count categories and compute average price
        cat_count: Dict[str, int] = {}
        cat_avg_price: Dict[str, List[float]] = {}
        for item in history:
            c = item.get("category", "other").lower()
            cat_count[c] = cat_count.get(c, 0) + 1
            cat_avg_price.setdefault(c, []).append(float(item.get("price", 0)))

        # Most frequent category
        most_freq = max(cat_count.items(), key=lambda kv: kv[1])[0]

        # Heuristic: recommend adjacent category
        adjacency_map = {
            "electronics": "accessories",
            "sportswear": "health",
            "books": "stationery",
            "groceries": "home",
            "other": "electronics",
        }

        recommended = adjacency_map.get(most_freq, "electronics")

        # If user bought headphones and shoes, maybe recommend wearable tech or sports accessories
        justification = (
            f"User frequently buys items in {most_freq}. Based on that, recommending {recommended} which complements their purchases (e.g., wearable tech for sports, accessories for electronics)."
        )

        return recommended, justification

    def _call_llm(self, user_profile: Dict[str, Any]) -> Dict[str, str]:
        # This is a stub. Replace with real LLM integration if desired.
        # Return structure: {"category": "...", "justification": "..."}
        return {"category": "electronics", "justification": "(LLM stub) Recommending electronics."}


# -----------------------------
# Ranking and Search
# -----------------------------

def infer_price_range_from_history(history: List[Dict[str, Any]]) -> Tuple[float, float]:
    if not history:
        return 20.0, 200.0
    prices = [float(item.get("price", 0)) for item in history]
    avg = sum(prices) / len(prices)
    # range +/- 40% for flexibility
    return max(5.0, avg * 0.6), avg * 1.4


def score_product_for_user(product: Product, price_range: Tuple[float, float]) -> float:
    # Higher score = better match
    low, high = price_range
    price_score = 1.0 - (max(0.0, abs((product.price - (low + high) / 2))) / max(1.0, (high - low) / 2 + 1))
    rating_score = (product.rating / 5.0)
    pop_score = math.log(product.popularity + 1) / math.log(1000 + 1)
    # weighted sum
    return 0.5 * price_score + 0.35 * rating_score + 0.15 * pop_score


def top_n_products(products: List[Product], n: int, price_range: Tuple[float, float]) -> List[Product]:
    scored = [(score_product_for_user(p, price_range), p) for p in products]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored[:n]]


# -----------------------------
# Agent Orchestration
# -----------------------------

class ShoppingAgent:
    def __init__(self, catalog: Catalog, memory: MemoryStore, recommender: LLMRecommender):
        self.catalog = catalog
        self.memory = memory
        self.recommender = recommender

    def analyze_and_recommend(self, user_profile: Dict[str, Any]) -> Dict[str, Any]:
        # 1. Analyze history and recommend
        rec_cat, justification = self.recommender.recommend_category(user_profile)

        # 2. Search catalog
        candidates = self.catalog.find_by_category(rec_cat)

        # 3. Infer price range
        price_range = infer_price_range_from_history(user_profile.get("history", []))

        # 4. Rank top 3
        top3 = top_n_products(candidates, 3, price_range)

        # 5. Prepare structured response
        response = {
            "recommended_category": rec_cat,
            "justification": justification,
            "price_range": {"low": price_range[0], "high": price_range[1]},
            "top_products": [asdict(p) for p in top3],
        }
        return response

    def purchase(self, user_profile: Dict[str, Any], product_id: str) -> Dict[str, Any]:
        # find product
        product = next((p for p in self.catalog.products if p.id == product_id), None)
        if not product:
            return {"success": False, "message": f"Product {product_id} not found."}

        # simulate purchase by creating a transaction
        tx = Transaction(
            user_id=user_profile["user_id"],
            product_id=product.id,
            product_name=product.name,
            price=product.price,
            timestamp=datetime.datetime.utcnow().isoformat() + "Z",
        )

        # append to transactions
        self.memory.append_transaction(tx)

        # update user profile history
        new_entry = {"product": product.name, "category": product.category, "price": product.price}
        profile = self.memory.read_profile(user_profile["user_id"]) or user_profile
        profile_history = profile.get("history", [])
        profile_history.append(new_entry)
        profile["history"] = profile_history
        self.memory.write_profile(user_profile["user_id"], profile)

        return {"success": True, "message": "Purchase simulated successfully.", "transaction": asdict(tx)}


# -----------------------------
# Demo / Example Usage
# -----------------------------

def demo_flow(agent: ShoppingAgent, user_profile: Dict[str, Any]):
    print("\n--- Agent: Analyze & Recommend ---")
    analysis = agent.analyze_and_recommend(user_profile)
    print("Recommended category:", analysis["recommended_category"])  # small friendly output
    print("Justification:\n", analysis["justification"] if "justification" in analysis else analysis.get("justification"))
    print("Price range:", analysis["price_range"])  # user-centered price range
    print("Top products:")
    for p in analysis["top_products"]:
        print(f" - {p['id']}: {p['name']} (${p['price']}) (category: {p['category']})")

    if analysis["top_products"]:
        # pick the top one to purchase
        pick = analysis["top_products"][0]["id"]
        print("\n--- Agent: Simulated Purchase ---")
        result = agent.purchase(user_profile, pick)
        print(result["message"])
        if result.get("success"):
            print("Transaction:", result["transaction"]) 


# -----------------------------
# If run as a script
# -----------------------------

if __name__ == "__main__":
    # example user profile provided in the requirements
    user_profile_example = {
        "user_id": "A123",
        "history": [
            {"product": "Bluetooth headphones", "category": "electronics", "price": 120},
            {"product": "Running shoes", "category": "sportswear", "price": 80},
        ],
    }

    # setup components
    catalog = Catalog()
    memory = MemoryStore(filepath="memory.json", tx_filepath="transactions.json")

    # seed memory with the example profile if it's not there
    existing = memory.read_profile(user_profile_example["user_id"])
    if not existing:
        memory.write_profile(user_profile_example["user_id"], user_profile_example)

    # recommender in heuristic mode by default (safe without API keys)
    recommender = LLMRecommender(mode="heuristic")
    agent = ShoppingAgent(catalog, memory, recommender)

    demo_flow(agent, user_profile_example)

    print("\nDemo finished.\nMemory and transactions saved to memory.json and transactions.json respectively.")

In [ ]:
# el haye7sal kal aaty 
# e7na hana5od el user history we nedeeha lel agent
# then el agent hayshoof random category min el categories eli mosh mawgooden fel buyers history